## 3. ML Modeling

- In this notebook, we will illustrate how to train an XGBoost model with the diamonds dataset using [OSS XGBoost](https://xgboost.readthedocs.io/en). 
- We also show how to do inference and manage models via Model Registry.

### Import Libraries

In [ ]:
!pip install shap

In [ ]:
# Snowpark for Python
from snowflake.snowpark.version import VERSION
import snowflake.snowpark.functions as F
from snowflake.snowpark.context import get_active_session

# Snowflake ML
from snowflake.ml.registry import Registry
from snowflake.ml._internal.utils import identifier

# data science libs
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.model_selection import GridSearchCV

# misc
import json
import joblib
import cachetools

# warning suppresion
import warnings; warnings.simplefilter('ignore')

In [ ]:
-- Use the SAME warehouse, database, and schema as notebook 1
USE WAREHOUSE <your_warehouse_name>;
USE DATABASE <your_database_name>;
USE SCHEMA <your_schema_name>;

-- Verify
SELECT CURRENT_WAREHOUSE() AS WAREHOUSE, CURRENT_DATABASE() AS DATABASE, CURRENT_SCHEMA() AS SCHEMA, CURRENT_ROLE() AS ROLE;

In [ ]:
# Establish Secure Connection to Snowflake
session = get_active_session()

# Add a query tag to the session.
session.query_tag = {"origin":"sf_sit-is", 
                     "name":"e2e_ml_snowparkpython", 
                     "version":{"major":1, "minor":0,},
                     "attributes":{"is_quickstart":1}}
#session

### Load the data & preprocessing pipeline

**Make sure you filled in the USE DATABASE and USE SCHEMA statements at the beginning of this notebook!**

The table name below uses your current database/schema from those USE statements.

In [ ]:
# Load in the data
# Using current session context - it will use whatever database/schema you have active
diamonds_df = ("DB.SCHMA.DIAMONDS") # Use your database and schema here

In [ ]:
# Categorize all the features for modeling
CATEGORICAL_COLUMNS = ["CUT", "COLOR", "CLARITY"]
CATEGORICAL_COLUMNS_OE = ["CUT_OE", "COLOR_OE", "CLARITY_OE"] # To name the ordinal encoded columns
NUMERICAL_COLUMNS = ["CARAT", "DEPTH", "TABLE_PCT", "X", "Y", "Z"]

LABEL_COLUMNS = ['PRICE']
OUTPUT_COLUMNS = ['PREDICTED_PRICE']

In [ ]:
# Recreate the preprocessing pipeline (same as in previous notebook)
# We recreate instead of loading from stage to avoid requiring stage permissions

import numpy as np
import snowflake.ml.modeling.preprocessing as snowml
from snowflake.ml.modeling.pipeline import Pipeline

categories = {
    "CUT": np.array(["IDEAL", "PREMIUM", "VERY_GOOD", "GOOD", "FAIR"]),
    "CLARITY": np.array(["IF", "VVS1", "VVS2", "VS1", "VS2", "SI1", "SI2", "I1", "I2", "I3"]),
    "COLOR": np.array(['D', 'E', 'F', 'G', 'H', 'I', 'J']),
}

preprocessing_pipeline = Pipeline(
    steps=[
            (
                "OE",
                snowml.OrdinalEncoder(
                    input_cols=CATEGORICAL_COLUMNS,
                    output_cols=CATEGORICAL_COLUMNS_OE,
                    categories=categories,
                )
            ),
            (
                "MMS",
                snowml.MinMaxScaler(
                    clip=True,
                    input_cols=NUMERICAL_COLUMNS,
                    output_cols=NUMERICAL_COLUMNS,
                )
            )
    ]
)

### Build a simple open-source XGBoost Regression model

In [ ]:
# Split the data into train and test sets
diamonds_train_df, diamonds_test_df = diamonds_df.random_split(weights=[0.9, 0.1], seed=0)

# Run the train and test sets through the Pipeline object we defined earlier
train_df = preprocessing_pipeline.fit(diamonds_train_df).transform(diamonds_train_df)
test_df = preprocessing_pipeline.transform(diamonds_test_df)

# Convert to pandas dataframes to use OSS XGBoost
train_pd = train_df.select(CATEGORICAL_COLUMNS_OE+NUMERICAL_COLUMNS+LABEL_COLUMNS).to_pandas()
test_pd = test_df.select(CATEGORICAL_COLUMNS_OE+NUMERICAL_COLUMNS+LABEL_COLUMNS).to_pandas()

In [ ]:
# Define model config
regressor = XGBRegressor()

# Split train data into X, y
y_train_pd = train_pd.PRICE
X_train_pd = train_pd.drop(columns=['PRICE'])

# Train model
regressor.fit(X_train_pd, y_train_pd)

In [ ]:
# We can now get predictions
y_test_pred = regressor.predict(test_pd.drop(columns=['PRICE']))
y_train_pred = regressor.predict(train_pd.drop(columns=['PRICE']))

Let's analyze the results using Snowflake ML's MAPE.

In [ ]:
mape = mean_absolute_percentage_error(y_train_pd, y_train_pred)

In [ ]:
print(f"Mean absolute percentage error: {mape}")

### Now, let's use `scikit-learn`'s `GridSearchCV` function to find optimal model parameters

In [ ]:
parameters={
        "n_estimators":[100, 200, 500],
        "learning_rate":[0.1, 0.4]
}

xgb = XGBRegressor()
clf = GridSearchCV(xgb, parameters)
clf.fit(X_train_pd, y_train_pd)

In [ ]:
print(clf.best_estimator_)

We see that the best estimator has the following parameters: `n_estimators=500` & `learning_rate=0.4`.

We can also analyze the full grid search results.

In [ ]:
# Analyze grid search results
gs_results = clf.cv_results_
n_estimators_val = []
learning_rate_val = []
for param_dict in gs_results["params"]:
    n_estimators_val.append(param_dict["n_estimators"])
    learning_rate_val.append(param_dict["learning_rate"])
mape_val = gs_results["mean_test_score"]*-1

gs_results_df = pd.DataFrame(data={
    "n_estimators":n_estimators_val,
    "learning_rate":learning_rate_val,
    "mape":mape_val})

sns.relplot(data=gs_results_df, x="learning_rate", y="mape", hue="n_estimators", kind="line")

plt.show()

This is consistent with the `learning_rate=0.4` and `n_estimator=500` chosen as the best estimator with the lowest MAPE.

Now, let's predict and analyze the results from using the best estimator.

In [ ]:
from sklearn.metrics import mean_absolute_percentage_error

# Predict
opt_model = clf.best_estimator_
y_train_pred = opt_model.predict(train_pd.drop(columns=['PRICE']))

mape = mean_absolute_percentage_error(y_train_pd, y_train_pred)

print(f"Mean absolute percentage error: {mape}")

Let's save our optimal model and its metadata:


In [ ]:
optimal_model = clf.best_estimator_
optimal_n_estimators = clf.best_estimator_.n_estimators
optimal_learning_rate = clf.best_estimator_.learning_rate

optimal_mape = gs_results_df.loc[(gs_results_df['n_estimators']==optimal_n_estimators) &
                                 (gs_results_df['learning_rate']==optimal_learning_rate), 'mape'].values[0]

### Manage models using Model Registry

Now, with Snowflake ML's [Model Registry](https://docs.snowflake.com/en/developer-guide/snowpark-ml/snowpark-ml-mlops-model-registry), we have a Snowflake native model versioning and deployment framework. This allows us to log models, tag parameters and metrics, track metadata, create versions, and ultimately execute batch inference tasks in a Snowflake warehouse or deploy to a Snowpark Container Service.

First, we will log our models.

Refer to [this Medium post](https://medium.com/snowflake/whats-in-a-name-model-naming-versioning-in-snowpark-model-registry-b5f7105fd6f6) on best practices for model naming & versioning conventions.

In [ ]:
# Get sample input data to pass into the registry logging function
X = train_df.select(CATEGORICAL_COLUMNS_OE+NUMERICAL_COLUMNS).limit(100)

db = identifier._get_unescaped_name(session.get_current_database())
schema = identifier._get_unescaped_name(session.get_current_schema())

# Define model name
model_name = "DIAMONDS_PRICE_PREDICTION"

# Create a registry and log the model
native_registry = Registry(session=session, database_name=db, schema_name=schema)

# Let's first log the very first model we trained
model_ver = native_registry.log_model(
    model_name=model_name,
    version_name='V0',
    model=regressor,
    sample_input_data=X, # to provide the feature schema
    target_platforms={'WAREHOUSE'}
)

# Add evaluation metric
model_ver.set_metric(metric_name="mean_abs_pct_err", value=mape)

# Add a description
model_ver.comment = "This is the first iteration of our Diamonds Price Prediction model. It is used for demo purposes."

# Now, let's log the optimal model from GridSearchCV
model_ver2 = native_registry.log_model(
    model_name=model_name,
    version_name='V1',
    model=optimal_model,
    sample_input_data=X, # to provide the feature schema
    target_platforms={'WAREHOUSE'}
)

# Add evaluation metric
model_ver2.set_metric(metric_name="mean_abs_pct_err", value=optimal_mape)

# Add a description
model_ver2.comment = f"This is the second iteration of our Diamonds Price Prediction model \
                        where we performed hyperparameter optimization. \
                        Optimal n_estimators & learning_rate: {optimal_n_estimators}, {optimal_learning_rate}"

In [ ]:
# Let's confirm they were added
native_registry.get_model(model_name).show_versions()

We can see what the default model is when we have multiple versions with the same model name:


In [ ]:
native_registry.get_model(model_name).default.version_name

Now we can use the optimal model to perform inference.

In [ ]:
model_ver = native_registry.get_model(model_name).version('v1')
result_sdf2 = model_ver.run(test_df, function_name="predict")
result_sdf2.show()

You can also execute inference using SQL. To do this, we will use a SQL cell and reference our model's predict method via the model object's name.

In [ ]:
test_df.write.mode('overwrite').save_as_table('DIAMONDS_TEST')

In [ ]:
--- for any other version (for example V1 below):
WITH model_version_alias AS MODEL DIAMONDS_PRICE_PREDICTION VERSION v1 SELECT a.*, model_version_alias!predict(a.CUT_OE, a.COLOR_OE, a.CLARITY_OE, a.CARAT, a.DEPTH, a.TABLE_PCT, a.X, a.Y, a.Z)['output_feature_0'] as prediction from DIAMONDS_TEST a

### Model Explainability

Another thing we may want to look at to better understand the predictions are explanations on what the model considers most impactful when generating the predictions. To generate these explanations, we'll use the [built-in explainability function](https://docs.snowflake.com/en/developer-guide/snowflake-ml/model-registry/model-explainability) from Snowflake ML. 

Under the hood, this function is based on [Shapley values](https://towardsdatascience.com/the-shapley-value-for-ml-models-f1100bff78d1). During the training process, machine learning models infer relationships between inputs and outputs, and Shapley values are a way to attribute the output of a machine learning model to its input features. By considering all possible combinations of features, Shapley values measure the average marginal contribution of each feature to the model’s prediction. While computationally intensive, the insights gained from Shapley values are invaluable for model interpretability and debugging.

Let's calculate these explanations based on our optimal model now.

In [ ]:
mv_explanations = model_ver.run(train_df, function_name="explain")
#mv_explanations

Let's visualize these explanations since it's a bit hard to just interpret the values themselves.

In [ ]:
import shap

# Create a sample of 1000 records
test_pd = test_df.to_pandas()
test_pd_sample = test_pd.sample(n=1000, random_state = 100).reset_index(drop=True)

# Compute shapley values for each model
shap_pd = model_ver.run(test_pd_sample, function_name="explain")

We see that `CARAT` has the biggest impact on the prediction values (`PRICE`) followed by the `Y dimension`, `CLARITY`, and `COLOR`. This is what we observed in the data exploration phase in the previous notebook too when plotting `PRICE vs CARAT`.

Let's save our training data into a Snowflake table to illustrate how the SQL API version of this function can be also be used to generate feature explanations.

In [ ]:
train_df.write.mode('overwrite').save_as_table('DIAMONDS_TRAIN')

Now, we can call the SQL API by calling the model and the version we want to evaluate and generate those explanations.

In [ ]:
WITH mv AS MODEL "DIAMONDS_PRICE_PREDICTION" VERSION "V1"
SELECT * FROM DIAMONDS_TRAIN,
  TABLE(mv!"EXPLAIN"(
    CUT_OE,
    COLOR_OE,
    CLARITY_OE,
    CARAT,
    DEPTH,
    TABLE_PCT,
    X,
    Y,
    Z
  ));

Let's do some clean up now. **UNCOMMENT THE FOLLOWING LINES TO DELETE THE MODEL BEFORE RE-RUNNING THIS NOTEBOOK. Don't delete the model if you plan to set up the Streamlit app.**

In [ ]:
# Clean up
#native_registry.delete_model(model_name)

In [ ]:
# Confirm it was deleted
#native_registry.show_models()